# Strategy Refactor

In [ ]:
%load_ext autoreload
%autoreload 2

from fibsem import utils, acquire
from fibsem.milling import get_milling_stages, mill_stages
from fibsem.milling.strategy import register_strategy
from autolamella.protocol.validation import validate_protocol
from pprint import pprint 

# TODO: add to fibsem.milling.strategy.__init__.py for automatic registration
try:
    from adaptive_polish.adaptive_polishing_strategy import AdaptivePolishingStrategy, AdaptivePolishingConfig
    register_strategy(AdaptivePolishingStrategy)
except ImportError as e:
    pass

In [ ]:
# connect to microscope
microscope, settings = utils.setup_session()

# load new style protocol with adaptive-polish with validation
PROTOCOL_PATH = "/home/patrick/github/adaptive_polish/src/adaptive_polish/scripts/protocol-on-grid-adaptive-polish-dl-new.yaml"
protocol = validate_protocol(utils.load_protocol(protocol_path=PROTOCOL_PATH))

In [ ]:
# acquire reference images (required to register paths)
acquire.take_reference_images(microscope, settings.image)

milling_stages = get_milling_stages("mill_polishing", protocol["milling"])

strategy_config: AdaptivePolishingConfig = milling_stages[0].strategy.config
print("Adaptive Polishing Config:")
print(f"Milling Interval: {strategy_config.milling_interval}s")
print(f"Maximum Cycles: {strategy_config.max_milling_cycles}")
print(f"Model: {strategy_config.model_path}")


In [ ]:
# run milling stages
mill_stages(microscope=microscope, stages=milling_stages)